# Digitaler Zwilling Pipeline - Automatisierte HTC-Modellierung
Dieses Notebook enthält meine vollständige Daten-Pipeline zur Erstellung eines analytischen Digitalen Zwillings. Ziel ist es, aus umfangreichen Ansys-Simulationsdaten (mehrere Millionen Datenpunkte) präzise Modelle für den Wärmeübergangskoeffizienten (HTC) zu trainieren.

Das Skript ist so aufgebaut, dass es auch auf neue thermische Simulationen angewendet werden kann. Es durchläuft dabei automatisch die folgenden fünf Schritte:
1. **ETL-Pipeline:** Einlesen der rohen Ansys-Daten im `.csv`-Format und Konvertierung in speichereffiziente Parquet-Dateien.
2. **Outlier Filtering:** Bereinigung der Daten von numerischen Gittersingularitäten (Mesh-Fehler an den Linsenkanten) mittels eines 1%-Quantilsfilters.
3. **KI-Training (Symbolic Regression):** Nutzung von `PySR`, um analytische Gleichungen für den HTC zu generieren.
4. **Validierung:** Automatisierter Test der generierten Formeln anhand des gesamten Datensatzes zur Ermittlung des tatsächlichen Fehlers (MAE).
5. **Inferenz-Klasse (API):** Bereitstellung einer Python-Klasse, mit der die Modelle für Vorhersagen in Millisekundenbruchteilen abgefragt werden können, wodurch rechenintensive CFD-Simulationen ersetzt werden.

In [8]:
import pandas as pd
import numpy as np
import re
import sympy
import tqdm
from pathlib import Path
import tkinter as tk
from tkinter import filedialog
from pysr import PySRRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

## Schritt 1: ETL-Pipeline (CSV -> Parquet)
Im ersten Schritt lese ich alle vorhandenen `.csv`-Dateien gebündelt ein. Dabei ordne ich die einzelnen Simulationsfälle (Cases) direkt ihren physikalischen Gruppen zu (beispielsweise Heiz- oder Abkühlphasen / Heatup vs. Cooldown). Um das Skript für neue Projekte wiederzuverwenden, muss lediglich das Dictionary `gruppen_zuordnung` sowie die Zuweisungslogik an die neuen Cases angepasst werden.

In [ ]:
def erstelle_parquet_universell():
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    
    print("Bitte wähle den Hauptordner mit den Cases aus...")
    main_folder_path = filedialog.askdirectory(title="Wähle den Ordner 'Cases'")
    if not main_folder_path:
        print("Abbruch: Kein Ordner ausgewählt.")
        return
        
    main_folder = Path(main_folder_path)
    all_tables = []
    print(f"Lese CSVs aus '{main_folder.name}' ein (Dies kann kurz dauern)...")

    # 1. ALLE DATEN ROH EINLESEN (Noch ohne Gruppen)
    for file_pfad in main_folder.rglob("*.csv"):
        df_temp = pd.read_csv(file_pfad)
        case_folder = next(p for p in file_pfad.parents if p.name.startswith("Case")).name
        
        hit = re.search(r"area_Element([^_]+)_([^_]+)_time_(.*?)\.csv", file_pfad.name)
        if hit:
            linse = hit.group(1)
            surface_num = hit.group(2)
            time_val = float(hit.group(3))
            
            mapping = {'0':'bottom','1':'bottom','2':'top','3':'top','4':'lateral','5':'support'} if linse in ['1', '3'] else {'0':'bottom','1':'bottom','2':'top','3':'lateral','4':'support'}
                
            df_temp['case_id'] = case_folder
            df_temp['Linsen_art'] = linse
            df_temp['Surface'] = mapping.get(surface_num, 'unknown')
            df_temp['Time'] = time_val
            
            # HTC berechnen
            df_temp = df_temp[(df_temp['T'] != 0) & (df_temp['contact area'] != 0)]
            df_temp['HTC'] = df_temp['Q'] / (df_temp['contact area'] * df_temp['T'])
            df_temp = df_temp.drop(columns=['Q'])
            
            all_tables.append(df_temp)

    if not all_tables:
        print("Keine gültigen CSV-Dateien gefunden.")
        return

    df_final = pd.concat(all_tables, ignore_index=True)

    
    # 2. Clustering
    
    print("Starte KI-gestützte automatische Gruppierung...")
    
    # Für die Phase schauen wir uns die echte TEMPERATUR (T) an!
    df_trend_T = df_final.groupby(['case_id', 'Time'])['T'].mean().unstack(level=0)
    
    # Für die Gruppen (Similarity) schauen wir uns den HTC an
    df_trend_HTC = df_final.groupby(['case_id', 'Time'])['HTC'].mean().unstack(level=0)
    
    gruppen_zuordnung = {}
    phasen_zuordnung = {}
    gruppen_zaehler = {'Heatup': 1, 'Cooldown': 1}
    
    # Strengere Toleranz, da wir Durchschnittskurven vergleichen
    toleranz_mae = 10

    # A) PHASEN-ERKENNUNG (mit Temperatur)
    for case in df_trend_T.columns:
        # Temperatur ganz am Anfang vs. ganz am Ende
        t_start = df_trend_T[case].dropna().iloc[0]
        t_ende = df_trend_T[case].dropna().iloc[-1]
        
        # Wenn es am Ende wärmer ist -> Heatup!
        phasen_zuordnung[case] = 'Heatup' if t_ende > t_start else 'Cooldown'

    # B) Ähnliche Cases in Gruppen zusammenfassen
    for phase in ['Heatup', 'Cooldown']:
        cases_in_phase = [c for c, p in phasen_zuordnung.items() if p == phase]
        zugewiesen = []
        
        for case in cases_in_phase:
            if case in zugewiesen:
                continue
                
            aktuelle_gruppe = f"{phase}_Gruppe_{gruppen_zaehler[phase]}"
            gruppen_zuordnung[case] = aktuelle_gruppe
            zugewiesen.append(case)
            
            # Vergleiche diesen Case mit allen anderen
            for anderer_case in cases_in_phase:
                if anderer_case not in zugewiesen:
                    # MAE der HTC-Kurven berechnen
                    differenz = (df_trend_HTC[case] - df_trend_HTC[anderer_case]).abs().mean()
                    
                    if differenz <= toleranz_mae:
                        gruppen_zuordnung[anderer_case] = aktuelle_gruppe
                        zugewiesen.append(anderer_case)
                        
            gruppen_zaehler[phase] += 1

    print("\nAutomatische Zuweisung erfolgreich:")
    for c, g in sorted(gruppen_zuordnung.items()):
        print(f"   -> {c}: {g} (Erkannt als {phasen_zuordnung[c]})")


    # 3. ZUWEISUNG ANWENDEN UND SPEICHERN
    
    df_final['Physik_Gruppe'] = df_final['case_id'].map(gruppen_zuordnung)
    df_final['Phase'] = df_final['case_id'].map(lambda x: 1 if phasen_zuordnung[x] == 'Heatup' else 0)

    # Parquet direkt im aktuellen Ordner (wo das Notebook liegt) ablegen
    speicher_pfad = Path("./linsen_daten_roh.parquet")
    df_final.to_parquet(speicher_pfad, index=False)
    
    # .absolute() zeigt dir in der Konsole den genauen, vollen Pfad an
    print(f"\nFertig! Daten gespeichert unter:\n{speicher_pfad.absolute()}")

# Führe die Funktion aus
erstelle_parquet_universell()

Bitte wähle den Hauptordner mit den Cases aus...
📂 Lese CSVs aus 'Cases' ein (Dies kann kurz dauern)...
🤖 Starte KI-gestützte automatische Gruppierung...

✅ Automatische Zuweisung erfolgreich:
   -> Case1: Heatup_Gruppe_1 (Erkannt als Heatup)
   -> Case2: Cooldown_Gruppe_1 (Erkannt als Cooldown)
   -> Case3: Heatup_Gruppe_2 (Erkannt als Heatup)
   -> Case4: Cooldown_Gruppe_1 (Erkannt als Cooldown)
   -> Case5: Heatup_Gruppe_1 (Erkannt als Heatup)
   -> Case6: Heatup_Gruppe_3 (Erkannt als Heatup)
   -> Case7: Heatup_Gruppe_1 (Erkannt als Heatup)

💾 Fertig! Daten gespeichert unter:
c:\Users\benti\Documents\VS\Proj\Zeis_test\linsen_daten_roh.parquet


## Schritt 2: Numerisches Rauschen filtern (Mesh Outlier Removal)
In Strömungssimulationen treten an scharfen Geometriekanten (wie Lateral oder Top) häufig Gitterfehler auf, die zu physikalisch unrealistischen HTC-Spitzenwerten führen. Um die Qualität des Modelltrainings zu gewährleisten, wende ich hier einen Filter an. Für jede Fläche und jeden Case entferne ich gezielt die oberen und unteren 1 % der Datenpunkte (Extremwerte), um dieses numerische Rauschen aus den Simulationen zu eliminieren.

In [ ]:
if Path("linsen_daten_roh.parquet").exists() and not Path("linsen_daten_clean.parquet").exists():
    print("Lade rohe Daten für Outlier-Filterung...")
    df = pd.read_parquet("linsen_daten_roh.parquet")
    original_zeilen = len(df)
    
    # 1% - 99% Quantilsfilterung pro Fläche und Case
    untere_grenze = df.groupby(['case_id', 'Linsen_art', 'Surface'])['HTC'].transform(lambda x: x.quantile(0.01))
    obere_grenze = df.groupby(['case_id', 'Linsen_art', 'Surface'])['HTC'].transform(lambda x: x.quantile(0.99))
    
    maske_sauber = (df['HTC'] >= untere_grenze) & (df['HTC'] <= obere_grenze)
    df_clean = df[maske_sauber].copy()
    
    entfernt = original_zeilen - len(df_clean)
    print(f" Filterung abgeschlossen! Es wurden {entfernt} kaputte Knotenpunkte ({(entfernt/original_zeilen)*100:.2f}%) entfernt.")
    
    ziel_pfad = Path("./linsen_daten_clean.parquet")
    df_clean.to_parquet(ziel_pfad, index=False)
    print(f"Saubere Daten als '{ziel_pfad.absolute()}' gespeichert.")
else:
    print(" Übersprungen: 'linsen_daten_clean.parquet' existiert bereits.")

🧹 Lade rohe Daten für Outlier-Filterung...
 Filterung abgeschlossen! Es wurden 815964 kaputte Knotenpunkte (1.98%) entfernt.
Saubere Daten als 'c:\Users\benti\Documents\VS\Proj\Zeis_test\linsen_daten_clean.parquet' gespeichert.


## Schritt 3: KI-Training (Symbolic Regression mit PySR)
In diesem Abschnitt nutze ich Symbolische Regression, um die thermodynamischen Kurven der Simulation mathematisch abzubilden. Der Algorithmus (`PySR`) kombiniert dabei vordefinierte Basisfunktionen (`+`, `-`, `*`, `/`, `exp`, `sin`, `abs`, `log`).

Um Overfitting zu vermeiden und die Robustheit der Gleichungen zu sichern, definiere ich im Code spezifische physikalische Constraints (beispielsweise das Verbot von verschachtelten Exponentialfunktionen). Das Training erfolgt separat für die zuvor definierten Modellgruppen (z.B. `Heatup_Gruppe_1`), um case-spezifische physikalische Zusammenhänge optimal abzubilden.

In [ ]:
def trainiere_modelle(ziel_gruppe=None, max_iterations=250):
    """
    Trainiert die Modelle. 
    Wenn 'ziel_gruppe' angegeben ist, wird nur diese trainiert (Targeted Retraining).
    """
    print("Lade bereinigte Daten...")
    df = pd.read_parquet("linsen_daten_clean.parquet")
    
    # Physik-Features berechnen
    df['t_skaliert'] = df['Time'] / 18000.0
    df['r'] = np.sqrt(df['x']**2 + df['y']**2)
    features_pysr = ['t_skaliert', 'x', 'y', 'z', 'r']
    
    # Wir ermitteln dynamisch alle Gruppen in den Daten
    alle_gruppen = df['Physik_Gruppe'].unique()
    if ziel_gruppe:
        alle_gruppen = [ziel_gruppe] # Sniper-Modus aktiv
        
    linsen_liste = ['1', '2', '3']
    flaechen_liste = ['top', 'bottom', 'lateral', 'support']
    
    basis_ordner = Path("./Formeln_Master")
    basis_ordner.mkdir(parents=True, exist_ok=True)
    
    for gruppe in alle_gruppen:
        print(f"\n🚀 Starte Training für Gruppe: {gruppe}")
        formeln_für_gruppe = []
        
        alle_aufgaben = [(l, f) for l in linsen_liste for f in flaechen_liste]
        
    
        for linse, flaeche in tqdm(alle_aufgaben, desc=f"Trainiere {gruppe}", unit="Fläche"):
            
            maske = (df['Surface'] == flaeche) & (df['Linsen_art'] == linse) & (df['Physik_Gruppe'] == gruppe)
            df_gefiltert = df[maske].iloc[::5]
            
            if df_gefiltert.empty:
                continue
            
            phase_name = "Heatup" if df_gefiltert['Phase'].iloc[0] == 1 else "Cooldown"
            X = df_gefiltert[features_pysr]
            y = df_gefiltert['HTC']
            
            modell = PySRRegressor(
                procs=4,
                niterations=max_iterations,
                binary_operators=["+", "-", "*", "/"],
                unary_operators=["exp", "sin", "abs", "log"],
                nested_constraints={"exp": {"exp": 0}, "sin": {"sin": 0}, "log": {"log": 0}},
                maxsize=30,
                warmup_maxsize_by=0.1,
                constraints={'/': (-1, 5), 'exp': 3},
                model_selection="best",
                verbosity=0,
                batching=True,
                batch_size=1000,
                progress=False 
            )
            
            # Wir nutzen keine print()-Befehle in der Schleife mehr, 
            # damit wir den tqdm-Balken nicht kaputt machen!
            modell.fit(X, y)
            
            vorhersage = modell.predict(X)
            mae = mean_absolute_error(y, vorhersage)
            
            ergebnis_zeile = f"Linse {linse} | {flaeche} | {gruppe} ({phase_name}) (Fehler: ?% / MAE: {mae:.3f}): {str(modell.sympy())}"
            formeln_für_gruppe.append(ergebnis_zeile)
                
        # In .txt abspeichern
        datei_pfad = basis_ordner / f"analytische_formeln_{gruppe}.txt"
        with open(datei_pfad, "w", encoding="utf-8") as f:
            for z in formeln_für_gruppe:
                f.write(z + "\n")
        print(f"Formeln für {gruppe} gesichert.")

## Schritt 4: Modellauswertung & Validierung
Da für das Training aus Performancegründen nur ein ausgedünnter Datensatz (ca. 20 % der Daten) verwendet wird, erfolgt in diesem Schritt die finale Validierung. Der Code evaluiert die generierten Formeln ungeschönt anhand von 100 % der bereinigten Datenpunkte aus der `linsen_daten_clean.parquet`. Hierbei berechne ich den tatsächlichen mittleren absoluten Fehler (Echter MAE), um sicherzustellen, dass das Modell generalisiert und die Physik verstanden hat.

In [ ]:
def evaluiere_zwilling():
    print("Lade bereinigte Parquet-Daten...")
    df = pd.read_parquet("linsen_daten_clean.parquet")
    df['t_skaliert'] = df['Time'] / 18000.0
    df['r'] = np.sqrt(df['x']**2 + df['y']**2)
    
    formel_ordner = Path("./Formeln_Master")
    formeln_daten = []
    
    if not formel_ordner.exists():
        print("Keine Formeln gefunden! Bitte trainiere das Modell zuerst.")
        return
        
    for datei in formel_ordner.glob("analytische_formeln_*Gruppe*.txt"):
        with open(datei, "r", encoding="utf-8") as f:
            for zeile in f:
                match = re.search(r"Linse (\d+)\s*\|\s*(\w+)\s*\|\s*([a-zA-Z0-9_]+)\s*\((Heatup|Cooldown)\).*MAE:\s*([0-9\.\+eE\-]+)\):\s*(.*)", zeile)
                if match:
                    formeln_daten.append({
                        'linse': match.group(1), 'flaeche': match.group(2),
                        'gruppe': match.group(3), 'phase_str': match.group(4),
                        'train_mae': float(match.group(5)), 'formel': match.group(6).strip()
                    })
                    
    test_ergebnisse = []
    print(f"\n Evaluiere {len(formeln_daten)} Master-Formeln auf dem Gesamt-Datensatz...")
    
    for fd in formeln_daten:
        maske = (df['Surface'] == fd['flaeche']) & (df['Linsen_art'] == fd['linse']) & (df['Physik_Gruppe'] == fd['gruppe'])
        df_test = df[maske].copy()
        if df_test.empty: continue
        
        try:
            expr = sympy.sympify(fd['formel'])
            symbole = tuple(expr.free_symbols)
            funktion = sympy.lambdify(symbole, expr, modules="numpy")
            inputs = [df_test[str(sym)] for sym in symbole]
            
            vorhersage = funktion(*inputs)
            if isinstance(vorhersage, (int, float)): vorhersage = np.full(len(df_test), vorhersage)
                
            echter_mae = mean_absolute_error(df_test['HTC'], vorhersage)
            
            test_ergebnisse.append({
                'Gruppe': fd['gruppe'], 'Phase': fd['phase_str'], 'Linse': fd['linse'], 'Surface': fd['flaeche'],
                'Trainings_MAE': round(fd['train_mae'], 3), 'Echter_MAE': round(echter_mae, 3),
                'Abweichung': round(abs(fd['train_mae'] - echter_mae), 3)
            })
        except Exception as e:
            print(f" Fehler bei Auswertung: {e}")
            
    if test_ergebnisse:
        df_bericht = pd.DataFrame(test_ergebnisse)
        df_bericht = df_bericht.sort_values(by='Echter_MAE', ascending=False)
        print(f"\n GLOBALTER DURCHSCHNITTS-FEHLER: {df_bericht['Echter_MAE'].mean():.2f} W/m²K\n")
        print(df_bericht.head(10).to_string(index=False))
        df_bericht.to_csv("Master_Testbericht.csv", index=False, sep=";", decimal=",")
        print("\n Kompletter Testbericht als 'Master_Testbericht.csv' gespeichert.")

evaluiere_zwilling() # Auskommentiert, bis Formeln vorliegen

Lade bereinigte Parquet-Daten...

🧪 Evaluiere 48 Master-Formeln auf dem Gesamt-Datensatz...

 GLOBALTER DURCHSCHNITTS-FEHLER: 46.93 W/m²K

         Gruppe  Phase Linse Surface  Trainings_MAE  Echter_MAE  Abweichung
Heatup_Gruppe_2 Heatup     2 support        353.313     353.278       0.035
Heatup_Gruppe_1 Heatup     2 support        306.880     306.233       0.647
Heatup_Gruppe_3 Heatup     2 support        282.776     283.334       0.558
Heatup_Gruppe_3 Heatup     1 support        211.718     210.720       0.998
Heatup_Gruppe_2 Heatup     1 support        178.109     177.239       0.870
Heatup_Gruppe_1 Heatup     1 support        154.159     154.631       0.472
Heatup_Gruppe_1 Heatup     3 lateral         69.847      70.041       0.194
Heatup_Gruppe_3 Heatup     3 lateral         68.407      68.943       0.536
Heatup_Gruppe_2 Heatup     3 lateral         68.110      68.234       0.124
Heatup_Gruppe_3 Heatup     3 support         64.645      64.392       0.253

 Kompletter Testbericht 

## Schritt 5: Die Inferenz-Klasse (Praxisanwendung)
Den Abschluss des Notebooks bildet eine objektorientierte Python-Klasse, die als API für die Anwendung im Unternehmen dient. Der Code liest die generierten Formeln initial ein und kompiliert sie mithilfe von `Numpy` für maximale Ausführungsgeschwindigkeit. Dadurch ist es möglich, den HTC-Wert für eine beliebige Linsenkoordinate in Millisekundenbruchteilen zu berechnen, was die Notwendigkeit erneuter und zeitaufwendiger CFD-Simulationen für diese Anwendungsfälle hinfällig macht.

In [7]:
class DigitalerZwillingHTC:
    def __init__(self, formel_ordner_pfad):
        self.formel_ordner = Path(formel_ordner_pfad)
        self.modelle = {}
        if self.formel_ordner.exists():
            self._lade_und_kompiliere_formeln()
        else:
            print(" Hinweis: Formel-Ordner noch leer/nicht vorhanden.")

    def _lade_und_kompiliere_formeln(self):
        print("Bootvorgang Digitaler Zwilling: Lade Formeln...")
        for datei in self.formel_ordner.glob("analytische_formeln_*Gruppe*.txt"):
            with open(datei, "r", encoding="utf-8") as f:
                for zeile in f:
                    match = re.search(r"Linse (\d+)\s*\|\s*(\w+)\s*\|\s*([a-zA-Z0-9_]+)\s*\((Heatup|Cooldown)\).*MAE:\s*[0-9\.\+eE\-]+\):\s*(.*)", zeile)
                    if match:
                        schluessel = f"{match.group(3)}_Linse{match.group(1)}_{match.group(2)}"
                        try:
                            expr = sympy.sympify(match.group(5).strip())
                            symbole = tuple(expr.free_symbols)
                            funktion = sympy.lambdify(symbole, expr, modules="numpy")
                            self.modelle[schluessel] = {'funktion': funktion, 'benoetigte_symbole': [str(s) for s in symbole]}
                        except Exception as e:
                            pass
        print(f"Bereit! {len(self.modelle)} Formeln im RAM kompiliert.\n")

    def predict_htc(self, gruppe, linse, flaeche, time, x, y, z):
        schluessel = f"{gruppe}_Linse{linse}_{flaeche}"
        if schluessel not in self.modelle:
            raise ValueError(f"Formel für '{schluessel}' nicht gefunden!")
            
        modell_info = self.modelle[schluessel]
        t_skaliert = time / 18000.0
        r = np.sqrt(x**2 + y**2)
        
        werte_dict = {'t_skaliert': t_skaliert, 'x': x, 'y': y, 'z': z, 'r': r}
        inputs = [werte_dict[sym] for sym in modell_info['benoetigte_symbole']]
        
        vorhersage = modell_info['funktion'](*inputs)
        return float(vorhersage[0]) if isinstance(vorhersage, np.ndarray) else float(vorhersage)

# --- BEISPIEL-NUTZUNG ---
"""
zwilling = DigitalerZwillingHTC("./Formeln_Master")
if zwilling.modelle:
    htc_wert = zwilling.predict_htc(
        gruppe="Heatup_Gruppe_3", linse="2", flaeche="lateral",
        time=500, x=0.05, y=-0.02, z=0.01
    )
    print(f" Berechneter HTC-Wert: {htc_wert:.2f} W/m²K")
"""

'\nzwilling = DigitalerZwillingHTC("./Formeln_Master")\nif zwilling.modelle:\n    htc_wert = zwilling.predict_htc(\n        gruppe="Heatup_Gruppe_3", linse="2", flaeche="lateral",\n        time=500, x=0.05, y=-0.02, z=0.01\n    )\n    print(f" Berechneter HTC-Wert: {htc_wert:.2f} W/m²K")\n'